# Parihaka facies: 5-seed ensemble on Kaggle
**Before running:** in the right-hand sidebar under *Session options*, set **Accelerator** to a GPU (T4 x2 is fine; the code uses one GPU)
and turn **Internet** on. Both require a phone-verified Kaggle account. Run the code cells top to bottom.

In [ ]:
!nvidia-smi -L

## 1. Code and dependencies

In [ ]:
%cd /kaggle/working
!test -d Wildcat-Neural-Network || git clone https://github.com/kten314/Wildcat-Neural-Network.git
%cd Wildcat-Neural-Network
!git pull -q
!pip install -q zarr segmentation-models-pytorch

## 2. Data (sizes should be about 1.6 GB and 6.9 MB)

In [ ]:
!mkdir -p data/raw/parihaka
!curl -fL -o data/raw/parihaka/parihaka_labels_train.npz "https://data.mendeley.com/public-files/datasets/gnvyh3msrj/files/0a0ac0af-10ab-47c3-83ac-f38fb0090d4a/file_downloaded"
!curl -fL -o data/raw/parihaka/parihaka_data_train.npz "https://data.mendeley.com/public-files/datasets/gnvyh3msrj/files/2777ae64-68e3-4cb0-aae4-588b30b7009c/file_downloaded"
!ls -lh data/raw/parihaka
!file data/raw/parihaka/*.npz

## 3. Convert to Zarr (about 8 minutes)

In [ ]:
!python src/convert_parihaka.py

## 4. Quick check of the evaluation pipeline (about 5 minutes)
Scores mean nothing here. If this step fails, stop and send the error before spending time on training.

In [ ]:
!python src/train_unet.py --epochs 1 --steps-per-epoch 10 --tag pipecheck --figs 0
!python src/ensemble_eval.py --tag pipecheck --figs 1

## 5. Train five ensemble members
Seeds 0 to 4, 30 epochs each. Skips seeds whose checkpoint already exists. Kaggle GPU time counts against your weekly quota.

In [ ]:
import os
for s in range(5):
    if os.path.exists(f"checkpoints/ens_seed{s}.pt"):
        print(f"seed {s} already trained, skipping")
    else:
        print(f"===== training seed {s} =====")
        get_ipython().system(f"python src/train_unet.py --epochs 30 --steps-per-epoch 100 --batch 16 --tag ens --seed {s} --figs 0")

## 6. Evaluate the ensemble

In [ ]:
!python src/ensemble_eval.py --tag ens --figs 3

## 7. Look at the figures

In [ ]:
from IPython.display import Image, display
for name in ["ensemble_ens_distance", "ensemble_ens_uncertainty", "ensemble_ens_gallery"]:
    display(Image(f"results/figures/{name}.png"))

## 8. Package the results
This creates `ensemble_results.zip` (a few MB). Download it from the Output / file panel in the sidebar
(the file lives in `/kaggle/working/Wildcat-Neural-Network/`).

In [ ]:
!zip -qr ensemble_results.zip results/ensemble_ens.json results/figures/ensemble_ens_distance.png results/figures/ensemble_ens_uncertainty.png results/figures/ensemble_ens_gallery.png results/ens_seed*.json
!ls -lh ensemble_results.zip